In [1]:
import sys
sys.path.append("..")

from phoenix import synthesis, gates, utils
from phoenix import Circuit
import cirq
import numpy as np
from phoenix import models

In [13]:
tab = models.BSF(['YZZI', 'IXXX'], [0.1, 0.2])
t1 = tab.apply_cx(1, 2)
print(t1)
print(t1.paulis)

BSF(size=[2, 4], num_nonlocals=2, total_weight=4)
['YIZI', 'IXIX']


In [14]:
synthesis.simplification.heuristic_bsf_cost(tab), synthesis.simplification.heuristic_bsf_cost(t1)

(array(23.5), array(22.5))

In [16]:
synthesis.simplification.simplify_bsf(tab)

(BSF(size=[2, 4], num_nonlocals=1, total_weight=2),
 [(C(Y, X) @ (0, 1), BSF(size=[0, 4], num_nonlocals=0, total_weight=0)),
  (C(X, Z) @ (1, 2), BSF(size=[0, 4], num_nonlocals=0, total_weight=0))])

In [21]:
# following is CNOT-family Clifford
models.cliffords.TRANSFORM_TABLE_2Q

,"C(X, X)","C(X, Y)","C(X, Z)","C(Y, X)","C(Y, Y)","C(Y, Z)","C(Z, X)","C(Z, Y)","C(Z, Z)"
II,II,II,II,II,II,II,II,II,II
IX,IX,XX,XX,IX,YX,YX,IX,ZX,ZX
IY,XY,IY,XY,YY,IY,YY,ZY,IY,ZY
IZ,XZ,XZ,IZ,YZ,YZ,IZ,ZZ,ZZ,IZ
XI,XI,XI,XI,XX,XY,XZ,XX,XY,XZ
XX,XX,IX,IX,XI,ZZ,-ZY,XI,-YZ,YY
XY,IY,XY,IY,-ZZ,XI,ZX,YZ,XI,-YX
XZ,IZ,IZ,XZ,ZY,-ZX,XI,-YY,YX,XI
YI,YX,YY,YZ,YI,YI,YI,YX,YY,YZ
YX,YI,-ZZ,ZY,YX,IX,IX,YI,XZ,-XY


In [39]:
from itertools import product

_PAULIS_2Q = [''.join(pair) for pair in product(['I', 'X', 'Y', 'Z'], ['I', 'X', 'Y', 'Z'])]
results = []
for pauli in _PAULIS_2Q:
    print(pauli)
    P = models.cliffords.PAULIS_2Q[pauli]
    Q = gates.ISWAP.data @ P @ gates.ISWAP.data.conj().T
    for pauli_, P_ in models.cliffords.PAULIS_2Q.items():
        if np.allclose(Q, P_):
            results.append((pauli_, 0))
            break
        elif np.allclose(Q, -P_):
            results.append((pauli_, 1))
            break
    else:
        raise ValueError(f"Not found {pauli_} in Clifford table")


II
IX
IY
IZ
XI
XX
XY
XZ
YI
YX
YY
YZ
ZI
ZX
ZY
ZZ


In [40]:
results

[('II', 0),
 ('YZ', 0),
 ('XZ', 1),
 ('ZI', 0),
 ('ZY', 0),
 ('XX', 0),
 ('YX', 0),
 ('IY', 0),
 ('ZX', 1),
 ('XY', 0),
 ('YY', 0),
 ('IX', 1),
 ('IZ', 0),
 ('YI', 0),
 ('XI', 1),
 ('ZZ', 0)]

In [76]:
from itertools import product
import qiskit.quantum_info as qi
from copy import deepcopy
from typing import Tuple
import pandas as pd
from phoenix.models.cliffords import PAULIS_2Q, assemble_paulistr_with_sign

Hz = gates.H.data
Hy = gates.S.data @ gates.H.data

class Clifford2QiSWAP:
    def __init__(self, pauli_0: str, pauli_1: str):
        assert pauli_0 in ['X', 'Y', 'Z'] and pauli_1 in ['X', 'Y', 'Z']
        I = qi.Pauli('I')
        P0, P1 = qi.Pauli(pauli_0), qi.Pauli(pauli_1)
        self.name = 'iSWAP({}, {})'.format(pauli_0, pauli_1)
        self.pauli_0, self.pauli_1 = pauli_0, pauli_1
        self.data = gates.ISWAP.data
        if self.pauli_0 == 'X':
            self.data = np.kron(Hz, I) @ self.data @ np.kron(Hz, I).conj().T
        elif self.pauli_0 == 'Y':
            self.data = np.kron(Hy, I) @ self.data @ np.kron(Hy, I).conj().T
        if self.pauli_1 == 'X':
            self.data = np.kron(I, Hz) @ self.data @ np.kron(I, Hz).conj().T
        elif self.pauli_1 == 'Y':
            self.data = np.kron(I, Hy) @ self.data @ np.kron(I, Hy).conj().T
        self.ctrl, self.targ = None, None

    def __repr__(self) -> str:
        if self.ctrl is not None and self.targ is not None:
            return f'C({self.pauli_0}, {self.pauli_1}) @ ({self.ctrl}, {self.targ})'
        return self.name

    def on(self, ctrl: int, targ: int) -> 'Clifford2QiSWAP':
        cliff = deepcopy(self)
        cliff.ctrl, cliff.targ = ctrl, targ
        return cliff

    def transform(self, pauli: str) -> Tuple[str, int]:
        """Transformation effect on a pair of Pauli operators."""
        P = PAULIS_2Q[pauli]
        Q = self.data @ P @ self.data.conj().T
        for pauli_, P_ in PAULIS_2Q.items():
            if np.allclose(Q, P_):
                return pauli_, 0
            if np.allclose(Q, -P_):
                return pauli_, 1
            
            
_TRANSFORM_TABLE_2Q = {}
_PAULIS_2Q = [''.join(pair) for pair in product(['I', 'X', 'Y', 'Z'], ['I', 'X', 'Y', 'Z'])]
for pauli_0, pauli_1 in product(['X', 'Y', 'Z'], ['X', 'Y', 'Z']):
    cg = Clifford2QiSWAP(pauli_0, pauli_1)  # controlled gate
    _TRANSFORM_TABLE_2Q[cg.name] = [assemble_paulistr_with_sign(*cg.transform(pauli)) for pauli in _PAULIS_2Q]
TRANSFORM_TABLE_2Q = pd.DataFrame(_TRANSFORM_TABLE_2Q, index=_PAULIS_2Q)
CLIFFORD_2Q_SET = [Clifford2QiSWAP(pauli_0, pauli_1) for pauli_0, pauli_1 in product(['X', 'Y', 'Z'], ['X', 'Y', 'Z'])]


In [77]:
print(TRANSFORM_TABLE_2Q.to_markdown())

|    | iSWAP(X, X)   | iSWAP(X, Y)   | iSWAP(X, Z)   | iSWAP(Y, X)   | iSWAP(Y, Y)   | iSWAP(Y, Z)   | iSWAP(Z, X)   | iSWAP(Z, Y)   | iSWAP(Z, Z)   |
|:---|:--------------|:--------------|:--------------|:--------------|:--------------|:--------------|:--------------|:--------------|:--------------|
| II | II            | II            | II            | II            | II            | II            | II            | II            | II            |
| IX | XI            | -ZY           | -YZ           | YI            | -ZY           | XZ            | ZI            | -XY           | YZ            |
| IY | ZX            | XI            | -ZZ           | ZX            | YI            | -ZZ           | XX            | ZI            | -XZ           |
| IZ | -YX           | -YY           | XI            | XX            | XY            | YI            | YX            | YY            | ZI            |
| XI | IX            | IY            | IZ            | -YZ           | -YZ           | -YX    